# 最小 Dataset テンプレート

目的: 画像フォルダ内の .jpg を列挙し、PyTorch `Dataset` として (画像Tensor, 画像ID, (任意) 生アノテーション) を返す最小構成を用意。

段階的拡張予定:
1. 画像読み込みのみ (現在)
2. リサイズ / 正規化
3. JSON 座標のスケーリング & マップ生成
4. DataLoader 最適化, キャッシュ

このセル以降のコードセルを順に実行してください。

In [ ]:
import os
from pathlib import Path
from typing import List, Optional

import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.transforms import functional as F
import json

# =============== 設定 ===============
IMAGES_DIR = Path('../kuzushiji_recognition/char_sep_datas')  # 画像ディレクトリ
GT_JSON_PATH = Path('../kuzushiji_recognition/char_sep_datas/gt_json.json')    # アノテーションJSON (未使用でもロード例)

# =============== Dataset 最小実装 ===============
class SimpleImageDataset(Dataset):
    """最小の自作 Dataset テンプレート

    現段階: 画像をTensor化して (tensor, image_id) を返すのみ。
    後で: JSON 利用 / ターゲット生成 / リサイズ / 前処理 を追加予定。
    """
    def __init__(self, images_dir: Path, json_path: Optional[Path] = None, extensions: List[str] = None, transform=None):
        self.images_dir = Path(images_dir)
        self.transform = transform
        self.extensions = extensions or ['.jpg', '.jpeg', '.png']

        assert self.images_dir.exists(), f"画像ディレクトリが存在しません: {self.images_dir}"

        # 画像ファイル列挙
        self.image_paths = [p for p in sorted(self.images_dir.iterdir()) if p.suffix.lower() in self.extensions]
        if not self.image_paths:
            raise RuntimeError(f"画像が見つかりません: {self.images_dir}")

        # JSON (必要ならロード) - 今は保持のみ
        self.raw_json = None
        if json_path is not None and json_path.exists():
            with open(json_path, 'r', encoding='utf-8') as f:
                self.raw_json = json.load(f)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        path = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            tensor = self.transform(image)
        else:
            tensor = F.to_tensor(image)  # [0,1] float32
        image_id = path.stem
        return tensor, image_id

# =============== 動作確認 ===============
# インスタンス生成
simple_ds = SimpleImageDataset(IMAGES_DIR, GT_JSON_PATH)
print('Dataset 長さ:', len(simple_ds))

# サンプル取得
tensor, image_id = simple_ds[0]
print('1枚目 shape:', tensor.shape, 'ID:', image_id)

# DataLoader 例
dataloader = DataLoader(simple_ds, batch_size=2, shuffle=False)
batch_imgs, batch_ids = next(iter(dataloader))
print('バッチ画像 shape:', batch_imgs.shape)
print('バッチID:', batch_ids)
